# Preprocesamiento – Detección temprana de riesgo académico – UNCAus
**Doctorado en Informática – UTN FRRe/UNNE/UNAM**  
Curso: Introducción al Aprendizaje de Máquina  
Integrantes: Stella Gerzel · Juan Manuel Goya · Gustavo Ebel

---
Este notebook parte del dataset unificado del EDA y construye:
1. Las decisiones de descarte de columnas
2. Los 3 targets (uno por problema de clasificación)
3. El split estratificado 75/25
4. El pipeline de preprocesamiento (imputación + codificación)

**Ejecutar en orden con Shift+Enter o Entorno de ejecución → Ejecutar todo**

In [ ]:
# CELDA 1 — Librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
import io, warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 110
print('Librerías OK')

In [ ]:
# CELDA 2 — Subir y unificar los 4 archivos Excel
# (mismo código que en el notebook de EDA)
from google.colab import files
uploaded = files.upload()

def encontrar(nombres, *palabras):
    for n in nombres:
        if all(p.lower() in n.lower() for p in palabras):
            return n
    return None

nombres = list(uploaded.keys())
reglas = [
    (('sistem','2024'), 'Result_1_2', 'Sistemas y Organizaciones',        2024),
    (('sistem','2025'), 'Result_8_2', 'Sistemas y Organizaciones',        2025),
    (('algor', '2024'), 'Result_4_2', 'Algoritmos y Estructuras de Datos', 2024),
    (('algor', '2025'), 'Result_5_2', 'Algoritmos y Estructuras de Datos', 2025),
]

dfs = []
for palabras, hoja, materia, cohorte in reglas:
    archivo = encontrar(nombres, *palabras)
    if archivo is None:
        print(f'  ATENCION: no encontré archivo para {palabras}'); continue
    tmp = pd.read_excel(io.BytesIO(uploaded[archivo]), sheet_name=hoja)
    tmp.insert(0, 'materia', materia)
    tmp.insert(1, 'cohorte', cohorte)
    dfs.append(tmp)
    print(f'  OK  {materia} {cohorte}: {len(tmp)} registros')

df = pd.concat(dfs, ignore_index=True)
print(f'\nDataset unificado: {df.shape[0]} filas x {df.shape[1]} columnas')

---
## PASO 1 · Descarte de columnas según criterios del EDA

In [ ]:
# CELDA 3 — Definir columnas a descartar
#
# Criterio 1: completamente vacías o sin variabilidad → no aportan información
# Criterio 2: domicilio granular → alta cardinalidad sin sentido predictivo
# Criterio 3: texto libre → no codificables directamente
# Criterio 4: identificadores → no son variables predictoras
# Criterio 5: el target → se maneja por separado

# Detectar automáticamente vacías y sin variabilidad
df_w = df.copy()
for col in df_w.select_dtypes('object').columns:
    df_w[col] = df_w[col].replace(['-','_',' '], np.nan)

pct_miss     = df_w.isna().mean()
vacias       = pct_miss[pct_miss == 1.0].index.tolist()
sin_var      = df_w.columns[df_w.nunique() <= 1].tolist()

# Columnas a descartar por criterio manual (EDA)
domicilio    = ['procedencia_calle', 'procedencia_numero', 'procedencia_piso',
                'procedencia_departamento', 'procedencia_unidad',
                'procedencia_barrio', 'procedencia_codigo_postal']

texto_libre  = ['trabajo_tarea_descripcion', 'costeo_estudios_descripcion']

idioma_raro  = ['idioma_otro', 'nivel_idioma_otro']  # >90% faltantes

identificadores = ['alumno']   # anonimizado pero no predictivo
target_col      = ['nota']     # se maneja por separado

# NOTA: procedencia_localidad se conserva para transformarla en km_a_sede
# (pendiente: subir localidades_para_completar.csv con los km)

descartar = list(set(vacias + sin_var + domicilio + texto_libre +
                     idioma_raro + identificadores + target_col))

# Columnas que entran al modelo
columnas_modelo = [c for c in df_w.columns if c not in descartar]

print(f'Columnas descartadas:  {len(descartar)}')
print(f'Columnas al modelo:    {len(columnas_modelo)}')
print(f'\nColumnas que entran al modelo:')
for c in columnas_modelo:
    print(f'  - {c}')

---
## PASO 2 · Definir los 3 targets

In [ ]:
# CELDA 4 — Crear los 3 targets binarios
#
# Problema 1: Riesgo global    → A=0 vs R+U=1  (todas las 978 filas)
# Problema 2: Desaprobación    → A=0 vs R=1    (solo filas A y R: 730 filas)
# Problema 3: Abandono         → A=0 vs U=1    (solo filas A y U: 527 filas)

# Dataset base: solo columnas del modelo + nota
X_base = df_w[columnas_modelo].copy()

# Problema 1
y1 = df_w['nota'].map({'A': 0, 'R': 1, 'U': 1})

# Problema 2: filtrar solo A y R
mask_p2 = df_w['nota'].isin(['A', 'R'])
X_p2    = X_base[mask_p2].reset_index(drop=True)
y2      = df_w.loc[mask_p2, 'nota'].map({'A': 0, 'R': 1}).reset_index(drop=True)

# Problema 3: filtrar solo A y U
mask_p3 = df_w['nota'].isin(['A', 'U'])
X_p3    = X_base[mask_p3].reset_index(drop=True)
y3      = df_w.loc[mask_p3, 'nota'].map({'A': 0, 'U': 1}).reset_index(drop=True)

print('Distribución de cada target:')
print(f'\nProblema 1 – Riesgo global ({len(y1)} filas):')
print(y1.value_counts().rename({0:'Aprobó (A)', 1:'No aprobó (R+U)'}))
print(f'\nProblema 2 – Desaprobación ({len(y2)} filas):')
print(y2.value_counts().rename({0:'Aprobó (A)', 1:'Desaprobó (R)'}))
print(f'\nProblema 3 – Abandono ({len(y3)} filas):')
print(y3.value_counts().rename({0:'Aprobó (A)', 1:'Abandonó (U)'}))

---
## PASO 3 · Split estratificado 75/25

In [ ]:
# CELDA 5 — Split para los 3 problemas
#
# stratify=y → garantiza que la proporción de clases sea igual
# en train y test (estratificado)
# random_state=42 → hace el split reproducible: todos obtenemos el mismo resultado

X1_train, X1_test, y1_train, y1_test = train_test_split(
    X_base, y1, test_size=0.25, stratify=y1, random_state=42)

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X_p2, y2, test_size=0.25, stratify=y2, random_state=42)

X3_train, X3_test, y3_train, y3_test = train_test_split(
    X_p3, y3, test_size=0.25, stratify=y3, random_state=42)

print('Tamaños del split:')
print(f'\nP1 – Train: {len(X1_train)} filas | Test: {len(X1_test)} filas')
print(f'  Proporción clase 1 en train: {y1_train.mean():.1%} | en test: {y1_test.mean():.1%}')
print(f'\nP2 – Train: {len(X2_train)} filas | Test: {len(X2_test)} filas')
print(f'  Proporción clase 1 en train: {y2_train.mean():.1%} | en test: {y2_test.mean():.1%}')
print(f'\nP3 – Train: {len(X3_train)} filas | Test: {len(X3_test)} filas')
print(f'  Proporción clase 1 en train: {y3_train.mean():.1%} | en test: {y3_test.mean():.1%}')
print()
print('Si las proporciones de clase 1 son casi iguales en train y test → el estratificado funcionó.')

---
## PASO 4 · Pipeline de preprocesamiento

In [ ]:
# CELDA 6 — Construir el pipeline
#
# Todas las columnas del modelo son categóricas (object)
# → un solo preprocesador para todas
#
# SimpleImputer(strategy='most_frequent') → rellena faltantes con la moda
# OneHotEncoder(handle_unknown='ignore') → convierte categorías en columnas 0/1
#   handle_unknown='ignore' → si en test aparece una categoría nueva que no
#   estaba en train, la ignora en vez de romper

cols_cat = X_base.select_dtypes('object').columns.tolist()
cols_num = X_base.select_dtypes('number').columns.tolist()

print(f'Columnas categóricas (→ impute + one-hot): {len(cols_cat)}')
print(f'Columnas numéricas   (→ impute + escala):  {len(cols_num)}')
print()

# Preprocesador categórico: moda → one-hot
cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Preprocesador numérico: moda (para los faltantes si los hay)
num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent'))
])

preprocesador = ColumnTransformer([
    ('cat', cat_transformer, cols_cat),
    ('num', num_transformer, cols_num)
])

# Verificar que el preprocesador funciona sobre el train del P1
X1_train_prep = preprocesador.fit_transform(X1_train)
X1_test_prep  = preprocesador.transform(X1_test)   # solo transform, no fit

print(f'X1_train después del pipeline: {X1_train_prep.shape}')
print(f'X1_test  después del pipeline: {X1_test_prep.shape}')
print()
print('El número de columnas creció: cada categoría única se convirtió en su propia columna 0/1.')
print('Esto es normal — es el resultado del one-hot encoding.')

In [ ]:
# CELDA 7 — Verificación visual del split estratificado

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
colores = ['#2ECC71', '#E74C3C']
labels  = ['Aprobó (0)', 'Riesgo (1)']

for i, (ytrain, ytest, titulo) in enumerate([
    (y1_train, y1_test, 'P1: Riesgo global'),
    (y2_train, y2_test, 'P2: Desaprobación'),
    (y3_train, y3_test, 'P3: Abandono')
]):
    ax = axes[i]
    x     = np.arange(2)
    w     = 0.35
    train_vals = [ytrain.value_counts().get(0,0), ytrain.value_counts().get(1,0)]
    test_vals  = [ytest.value_counts().get(0,0),  ytest.value_counts().get(1,0)]
    ax.bar(x - w/2, train_vals, w, label='Train', color='#3498DB', alpha=0.8)
    ax.bar(x + w/2, test_vals,  w, label='Test',  color='#F39C12', alpha=0.8)
    ax.set_xticks(x); ax.set_xticklabels(['Clase 0', 'Clase 1'])
    ax.set_title(titulo, fontweight='bold')
    ax.legend()

plt.suptitle('Split estratificado: proporciones de clase similares en train y test',
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print('\nPreprocesamiento completo.')
print('Siguiente paso: entrenar los modelos (LogReg, Random Forest, XGBoost) sobre cada problema.')